In [3]:
%pip install statsmodels
%pip install sklearn


import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm

# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_squared_error

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [ ]:
df = pd.read_excel("/content/dataset_ipm_dummy.xlsx")
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
X = df.drop(columns=['ipm', 'ipm_tinggi', 'kategori_ipm'])
y = df['ipm']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
def best_subset(X, y):
    results = []

    for k in range(1, len(X.columns)+1):
        for combo in itertools.combinations(X.columns, k):
            X_model = sm.add_constant(X[list(combo)])
            model = sm.OLS(y, X_model).fit()

            results.append({
                'variables': combo,
                'adj_r2': model.rsquared_adj,
                'aic': model.aic,
                'bic': model.bic
            })

    results_df = pd.DataFrame(results)
    return results_df.sort_values(by='adj_r2', ascending=False)

best_models = best_subset(X_train, y_train)
best_models.head(10)

In [ ]:
def forward_selection(X, y):
    remaining = list(X.columns)
    selected = []
    current_score = 0

    while remaining:
        scores = []

        for candidate in remaining:
            X_model = sm.add_constant(X[selected + [candidate]])
            model = sm.OLS(y, X_model).fit()
            scores.append((model.rsquared_adj, candidate))

        scores.sort()
        best_new_score, best_candidate = scores.pop()

        if best_new_score > current_score:
            remaining.remove(best_candidate)
            selected.append(best_candidate)
            current_score = best_new_score
        else:
            break

    return selected

selected_vars = forward_selection(X_train, y_train)
print("Selected Variables:", selected_vars)

In [ ]:
X_final = sm.add_constant(X_train[selected_vars])
model_final = sm.OLS(y_train, X_final).fit()

print(model_final.summary())

In [ ]:
X_test_final = sm.add_constant(X_test[selected_vars])
y_pred = model_final.predict(X_test_final)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("MSE:", mse)
print("RMSE:", rmse)

# Check Multikolinearitas

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_temp = sm.add_constant(X)
vif = pd.DataFrame()
vif["Variable"] = X_temp.columns
vif["VIF"] = [variance_inflation_factor(X_temp.values, i) 
              for i in range(X_temp.shape[1])]

print(vif)

# Visualiasai Hasil Model

In [ ]:
plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test, y=y_pred)
plt.xlabel("Actual IPM")
plt.ylabel("Predicted IPM")
plt.title("Actual vs Predicted")
plt.show()